In [1]:
HOPSWORKS_PROJECT_NAME= "taxi_demand_predictor"

In [2]:
# reading the API key from the .env file into python script

import os
from dotenv import load_dotenv
from src.paths import PARENT_DIR

# load key-value pairs from .env file located in the parent directory
load_dotenv(PARENT_DIR / ".env")

HOPSWORKS_API_KEY= os.environ["HOPSWORKS_API_KEY"]

In [3]:
# fetching the historical raw data from the data source

from datetime import datetime
import pandas as pd
from src.data import load_raw_data

from_year= 2022
to_year= datetime.now().year
print(f"Downloading data from {from_year} to {to_year} ...")

rides_lst= []
for year in range(from_year, to_year + 1):

    # download data for the whole year 
    rides_one_year= load_raw_data(year)

    # append data for the year to the main dataframe
    rides_lst.append(rides_one_year)
rides= pd.concat(rides_lst)

File 2022-01 was already in local storage
File 2022-02 was already in local storage
File 2022-03 was already in local storage
File 2022-04 was already in local storage
File 2022-05 was already in local storage
File 2022-06 was already in local storage
File 2022-07 was already in local storage
File 2022-08 was already in local storage
File 2022-09 was already in local storage
File 2022-10 was already in local storage
File 2022-11 was already in local storage
File 2022-12 was already in local storage
File 2023-01 was already in local storage
File 2023-02 was already in local storage
File 2023-03 was already in local storage
File 2023-04 was already in local storage
File 2023-05 was already in local storage
File 2023-06 was already in local storage
File 2023-07 was already in local storage
File 2023-08 was already in local storage
File 2023-09 was already in local storage
File 2023-10 was already in local storage
File 2023-11 was already in local storage
File 2023-12 was already in local 

In [4]:
print(f"{len(rides)= }")

len(rides)= 201435049


In [5]:
# transform the raw data into time-series data
from src.data import transform_raw_data_into_ts_data
ts_data= transform_raw_data_into_ts_data(rides)

100%|██████████| 263/263 [00:26<00:00,  9.96it/s]


In [6]:
import hopsworks

In [7]:
# connect to the project in Hopsworks
project= hopsworks.login(
    project= HOPSWORKS_PROJECT_NAME,
    api_key_value= HOPSWORKS_API_KEY
)

2026-11-10 16:49:47,010 INFO: Initializing external client
2026-11-10 16:49:47,013 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/55417


In [9]:
# connect to the feature store
feature_store= project.get_feature_store()

In [12]:
# save data (backfill) into the feature store

# first: set the feature group name and version
FEATURE_GROUP_NAME= "time_series_hourly_feature_group"
FEATURE_GROUP_VERSION= 1

In [13]:
# second: create the feature group
feature_group= feature_store.get_or_create_feature_group(
    name= FEATURE_GROUP_NAME,
    version= FEATURE_GROUP_VERSION,
    description= "Time-series data at hourly frequency",
    primary_key= ["pickup_hour", "pickup_location_id"],
    event_time= "pickup_hour",
    time_travel_format="ICEBERG",
)

In [14]:
# third: save the data into the feature store
feature_group.insert(ts_data, write_options={"wait_for_job": False})

2026-11-10 16:54:09,105 INFO: Python Engine initialized.


Feature Group created successfully, explore it at 
https://eu-west.cloud.hopsworks.ai:443/p/55417/fs/43078/fg/73803


Uploading Dataframe: 100.00% |██████████| Rows 10945008/10945008 | Elapsed Time: 11:15 | Remaining Time: 00:00


Launching job: time_series_hourly_feature_group_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/55417/jobs/named/time_series_hourly_feature_group_1_offline_fg_materialization/executions


(Job('time_series_hourly_feature_group_1_offline_fg_materialization', 'PYSPARK'),
 None)